Review Sentiment ANN (Positive / Neutral / Negative)

In [1]:
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

Loading eviews from your database

In [3]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password="Netra@432",
    database="books_db1",
)
cursor = conn.cursor(dictionary=True)
cursor.execute("""
    SELECT review_text, ratings FROM reviews
    WHERE review_text IS NOT NULL AND TRIM(review_text) <> '' AND ratings IS NOT NULL
""")
df = pd.DataFrame(cursor.fetchall())
conn.close()

# Have a CSV instead (columns: review_text, ratings)? Use this line instead of the code above:
# df = pd.read_csv("reviews.csv")

df["ratings"] = df["ratings"].astype(float)
print("Number of reviews:", len(df))
if len(df) < 300:
    print("WARNING: very few reviews, the ANN will not be accurate. Add more reviews or use a bigger CSV.")

Number of reviews: 821


In [4]:
def rating_to_sentiment(r):
    if r <= 2:
        return "negative"
    elif r == 3:
        return "neutral"
    return "positive"

df["sentiment"] = df["ratings"].apply(rating_to_sentiment)

label_map = {"negative": 0, "neutral": 1, "positive": 2}     # this order is also used by the website
df["label"] = df["sentiment"].map(label_map)

df["sentiment"].value_counts()

sentiment
positive    505
negative    197
neutral     119
Name: count, dtype: int64

Clean text

In [5]:
def clean_text(text):
    text = str(text).lower().replace("’", "'")
    text = text.replace("won't", "will not").replace("can't", "can not")
    text = re.sub(r"n't\b", " not", text)        # didn't -> did not
    text = re.sub(r"[^a-z ]", " ", text)
    return " ".join(text.split())

df["clean_review"] = df["review_text"].apply(clean_text)
df[["review_text", "clean_review", "sentiment"]].head()

,review_text,clean_review,sentiment
0,nice,nice,positive
1,It was a nice book but not my type,it was a nice book but not my type,neutral
2,it was nice book,it was nice book,positive
3,One of the amazing books I have read,one of the amazing books i have read,positive
4,too much of fantasy...far from reality,too much of fantasy far from reality,neutral


In [6]:
x_train, x_test, y_train, y_test = train_test_split(
    df["clean_review"], df["label"].values,
    test_size=0.2, random_state=42, stratify=df["label"]
)

cv = CountVectorizer(max_features=3000, ngram_range=(1, 2))
x_train_cv = cv.fit_transform(x_train).toarray()
x_test_cv = cv.transform(x_test).toarray()
x_train_cv.shape

(656, 3000)

In [8]:
model = Sequential([
    Input(shape=(x_train_cv.shape[1],)),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(3, activation="softmax"),      # 3 parts (negative, neutral, positive)
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                      │ (None, 64)                  │         192,064 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 3)                   │             195 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 192,259 (751.01 KB)

 Trainable params: 192,259 (751.01 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = dict(zip(np.unique(y_train), weights))

history = model.fit(
    x_train_cv, y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight,
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
)

Epoch 1/15


In [ ]:
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="validation accuracy")
plt.xlabel("epoch")
plt.legend()
plt.show()

## 7. Evaluate on the test set

In [ ]:
pred = np.argmax(model.predict(x_test_cv), axis=1)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, labels=[0, 1, 2],
                            target_names=["negative", "neutral", "positive"], zero_division=0))

## 8. Try your own review

In [ ]:
names = ["negative", "neutral", "positive"]

def predict_sentiment(text):
    probs = model.predict(cv.transform([clean_text(text)]).toarray(), verbose=0)[0]
    return names[int(np.argmax(probs))], probs.round(3)

for t in ["I loved this book, could not put it down!",
          "Boring and way too long, I would not recommend it.",
          "It was okay, some good parts and some slow parts."]:
    print(t, "->", predict_sentiment(t))

## 9. Save for the website

This creates `sentiment_model.keras` and `count_vectorizer.pkl`. Copy **both** next to `app.py`.

In [ ]:
model.save("sentiment_model.keras")
with open("count_vectorizer.pkl", "wb") as f:
    pickle.dump(cv, f)
print("saved: sentiment_model.keras, count_vectorizer.pkl")